# 02 — Area–elevation–volume (AEV) curve (exploration)

Visual and numerical check of `python -m thwake baseline --step aev`
(method: `docs/methodology.md` §1.3, ADR 0006). **Exploration only:** all logic lives in
`src/thwake/volume.py` and `src/thwake/baseline.py`; this notebook reads the files in
`data/baseline/` (and, in the last section, Earth Engine for a map).

**Status: draft curve, not frozen** (the freeze is prompt 05).

What to check:
- both curves start at the riverbed and rise smoothly to the FSL (912 m a.s.l.);
- volume at FSL for each DEM against the design capacity (688 MCM) and the
  design-history range (681–825 MCM);
- how far apart the two DEMs are at each level (one input to volume uncertainty);
- the rim check: SRTM's basin overflows below FSL at pass (a) (reported, not re-masked).

In [ ]:
import json

import pandas as pd
from IPython.display import Image

from thwake.baseline import aev_settings
from thwake.config import load_config
from thwake.volume import read_aev_csv

cfg = load_config()
s = aev_settings(cfg)
curves = read_aev_csv(s.aev_path)
meta = json.loads(s.metadata_path.read_text())
glo, srtm = curves["copernicus_glo30"], curves["srtm"]

## Summary at FSL

In [ ]:
pd.DataFrame(
    {
        d["label"]: {
            "DEM acquired": d["acquired"],
            "vertical datum": d["vertical_datum"],
            "lowest elevation in mask (m)": d["min_elevation_in_mask_m"],
            "mask area above FSL in this DEM (km²)": d["mask_area_above_fsl_km2"],
            "area at FSL (km²)": d["area_at_fsl_km2"],
            "volume at FSL (MCM)": d["volume_at_fsl_mcm"],
            "vs design 688 MCM (%)": d["capacity_check"]["vs_design_pct"],
            "within 681–825 MCM": d["capacity_check"]["within_design_history"],
            "dV/dh at FSL (MCM per m)": d["dvolume_dlevel_at_fsl_mcm_per_m"],
            "first rim overflow (m)": d["rim"]["first_overflow_m_asl"],
        }
        for d in meta["dems"].values()
    }
)

In [ ]:
Image(filename=str(s.figure_path))

## How far apart are the DEMs?

Volume difference (SRTM − GLO-30) at the same level, and the level and volume each curve
gives for the same measured area (this is how Phase 2 converts satellite water area).

In [ ]:
levels = [850, 860, 870, 880, 890, 900, 905, 910, 912]
pd.DataFrame(
    {
        "GLO-30 volume (MCM)": [glo.volume_at_level(h) for h in levels],
        "SRTM volume (MCM)": [srtm.volume_at_level(h) for h in levels],
    },
    index=pd.Index(levels, name="level_m_asl"),
).assign(
    diff_mcm=lambda t: t["SRTM volume (MCM)"] - t["GLO-30 volume (MCM)"],
    diff_pct=lambda t: 100 * t["diff_mcm"] / t["GLO-30 volume (MCM)"],
).round(1)

In [ ]:
areas = [2, 5, 10, 15, 20, 25, 29, 30]
rows = []
for a in areas:
    row = {"area_km2": a}
    for name, c in curves.items():
        try:
            row[f"{name} level (m)"] = round(c.level_at_area(a), 2)
            row[f"{name} volume (MCM)"] = round(c.volume_at_area(a), 1)
        except ValueError:  # area beyond this curve's range
            row[f"{name} level (m)"] = row[f"{name} volume (MCM)"] = None
    rows.append(row)
pd.DataFrame(rows).set_index("area_km2")

## Rim check and DEM difference

Each DEM was flood-filled with only the wall as a barrier. GLO-30 first overflows at ≈913 m
(pass (a), Makueni side); SRTM overflows at the same pass at ≈910 m, *below* FSL. The AEV
curve still uses the GLO-30 mask for both DEMs (open question 20).

In [ ]:
for d in meta["dems"].values():
    print(d["label"], json.dumps(d["rim"], indent=1))
print(json.dumps(meta["dem_comparison"], indent=1))

## Map (needs Earth Engine)

SRTM − GLO-30 elevation difference inside the max extent (blue: SRTM lower), with the wall
axis and the rim pass where each DEM overflows.

In [ ]:
import ee
import geemap

from thwake import ee_auth
from thwake.baseline import extent_settings
from thwake.collections import copernicus_dem, srtm_dem

ee_auth.initialize(cfg)
ext = extent_settings(cfg)
mask = geemap.geojson_to_ee(json.loads(ext.max_extent_path.read_text()))
region = mask.geometry().bounds().buffer(2000)
glo_dem, _ = copernicus_dem(cfg, region)
srtm_img, _ = srtm_dem(cfg)
diff = srtm_img.subtract(glo_dem).clipToCollection(mask)

Map = geemap.Map(center=(ext.wall_axis[0][1], ext.wall_axis[0][0]), zoom=12)
Map.addLayer(ee.Terrain.hillshade(glo_dem), {}, "Hillshade (GLO-30)")
Map.addLayer(diff, {"min": -6, "max": 6, "palette": ["2166ac", "f7f7f7", "b2182b"]}, "SRTM − GLO-30 (m)")
Map.addLayer(mask.style(color="00ffff", fillColor="00000000"), {}, "Max extent")
Map.addLayer(ee.Geometry.LineString([list(p) for p in ext.wall_axis]), {"color": "ff00ff"}, "Wall axis")
for d in meta["dems"].values():
    for p in d["rim"].get("overflow_passes", []):
        Map.addLayer(ee.Geometry.Point([p["lon"], p["lat"]]), {"color": "ff0000"}, f"{d['label']} overflow pass")
Map